In [2]:
import pandas as pd
import numpy as np
import re

# Load the dataset
df = pd.read_csv('../data/cleaned_data.csv', encoding='utf-8')

print(f"Dataset Loaded Successfully! Total Rows: {df.shape[0]}")

Dataset Loaded Successfully! Total Rows: 13616


In [ ]:
missing_markers = ['N/A', 'n/a', 'None', 'none', 'NA', '']

df.replace(missing_markers, np.nan, inplace=True)

print("Missing markers successfully converted to NaN values!")

Missing markers successfully converted to NaN values!


In [ ]:
def extract_numeric_stipend(stipend_value):
    if pd.isna(stipend_value):
        return np.nan
    
    stipend_str = str(stipend_value).strip()
    
    cleaned_string = re.sub(r'[^\d\-]', '', stipend_str)
    
    if not cleaned_string:
        return np.nan
    
    if '-' in cleaned_string:
        parts = cleaned_string.split('-')
        try:
            low = float(parts[0])
            high = float(parts[1])
            return (low + high) / 2
        except:
            return np.nan
            
    else:
        try:
            return float(cleaned_string)
        except:
            return np.nan

print("Stipend parsing function successfully defined!")

Stipend parsing function successfully defined!


In [ ]:
df['cleaned_stipend_monthly'] = df['stipend'].apply(extract_numeric_stipend)

df[['stipend', 'cleaned_stipend_monthly']].head(5)

,stipend,cleaned_stipend_monthly
0,"₹ 1,000 - 1,100 /month",1050.0
1,"₹ 10,000 - 15,000 /month",12500.0
2,"₹ 5,000 - 10,000 /month",7500.0
3,"₹ 9,500 - 16,000 /month",12750.0
4,"₹ 11,000 - 17,000 /month",14000.0


In [ ]:
text_columns_to_fill = ['skills', 'description', 'company_about', 'recruiter_email', 'company_website']

for col in text_columns_to_fill:
    df[col] = df[col].fillna('Missing')

print("Remaining missing values for our text columns:")
print(df[text_columns_to_fill].isnull().sum())

Remaining missing values for our text columns:
skills             0
description        0
company_about      0
recruiter_email    0
company_website    0
dtype: int64


In [ ]:
df['company'] = df['company'].astype(str).str.strip()
df['title'] = df['title'].astype(str).str.strip()

print("Company names and Job titles successfully stripped of messy whitespaces!")

Company names and Job titles successfully stripped of messy whitespaces!


In [ ]:
df.to_csv('../data/processed_cleaned_data.csv', index=False)

print("Clean dataset has been saved to '../data/processed_cleaned_data.csv'")

Clean dataset has been saved to '../data/processed_cleaned_data.csv'


In [ ]:
def labeling_engine(row):
    score = 0
    
    website = str(row['company_website']).lower().strip()
    about = str(row['company_about']).lower().strip()
    email = str(row['recruiter_email']).lower().strip()
    
    if website == 'missing' and about == 'missing':
        score += 10
    
    if email != 'missing':
        if any(g_dom in email for g_dom in ['@yahoo', '@outlook', '@hotmail', '@live']):
            score += 15

    title = str(row['title']).lower()
    title_scam_words = [
        'typing job', 'form filling', 'data entry operator', 'earn money', 
        'simple online job', 'copy paste', 'sms sending', 'part time online'
    ]
    if any(tw in title for tw in title_scam_words):
        score += 25

    desc = str(row['description']).lower()
    
    money_triggers = ['fee', 'fees', 'charge', 'charges', 'deposit', 'payment', 'amount', 'cost', 'money']
    scam_contexts = ['registration', 'training', 'security', 'documentation', 'processing', 'laptop', 'materials', 'verification', 'application']
    
    has_monetary_trap = any(m in desc for m in money_triggers) and any(c in desc for c in scam_contexts)
    if has_monetary_trap:
        score += 45  # Increased to 45 so monetary traps trigger scam classification instantly!
        
    direct_scam_phrases = [
        'whatsapp directly', 'investment', 'daily profit', 'pay to work', 
        'income source', 'telegram link', 'crypto wallet', 'task based earning'
    ]
    if any(phrase in desc for phrase in direct_scam_phrases):
        score += 25

    stipend = row['cleaned_stipend_monthly']
    if pd.notna(stipend):
        if stipend >= 40000:
            score += 30
        elif stipend > 25000 and any(d in str(row['duration']).lower() for d in ['1 week', '2 weeks', '1 month']):
            score += 25

    return 1 if score >= 45 else 0

print("Sscoring engine compiled successfully!")

Sscoring engine compiled successfully!


In [ ]:
df['is_scam'] = df.apply(labeling_engine, axis=1)

print("--- CLASS DISTRIBUTION ---")
print(df['is_scam'].value_counts())

print("\n--- PERCENTAGE BREAKDOWN ---")
print(df['is_scam'].value_counts(normalize=True) * 100)

--- CLASS DISTRIBUTION ---
is_scam
0    10607
1     3009
Name: count, dtype: int64

--- PERCENTAGE BREAKDOWN ---
is_scam
0    77.900999
1    22.099001
Name: proportion, dtype: float64


In [41]:
df.to_csv('../data/processed_cleaned_data.csv', index=False)

print("Progress successfully saved to '../data/processed_cleaned_data.csv'!")

Progress successfully saved to '../data/processed_cleaned_data.csv'!
